# Notebook 06 — RQ4: Risk & Drawdown Control

Two-zone kill-switch with RF classifier and DRL (PPO-proxy) position sizing. CVaR, bootstrap confidence intervals, Kupiec VaR test. Tests H0: drawdowns are random and uncontrollable.

**QM640 Data Analytics Capstone · Walsh College · Anupam K Mitra · May 2026**

---
> **Standalone notebook** — all code is self-contained. Run cells top-to-bottom. Data is downloaded automatically on first run.

## 1. Setup

In [ ]:
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import yfinance as yf, time
warnings.filterwarnings('ignore')
%matplotlib inline

# ── India parameters ────────────────────────────────────────────
TICKERS   = ['^NSEI','TCS.NS','INFY.NS','HDFCBANK.NS','SBIN.NS',
             'SUNPHARMA.NS','HINDUNILVR.NS','ONGC.NS','COALINDIA.NS',
             'TITAN.NS','MARUTI.NS','LT.NS','USDINR=X','GC=F','CL=F','SPY']
START, END     = '2010-01-01','2026-04-30'
RF_ANNUAL      = 0.065   # RBI repo
COST_BPS       = 20.0
BENCHMARK      = '^NSEI'
# Two-zone parameters (tuned for India)
DRL_LAMBDA     = 0.8
SOFT_ATTEN     = 0.25
HARD_STOP_P    = 0.82
DEAD_ZONE      = 0.45
W_MIN          = 0.25
DD_THRESHOLD   = -0.10

print('Config loaded — India NSE universe')

## 2. Download Data

In [ ]:
dfs = {}
for t in TICKERS:
    time.sleep(0.3)
    try:
        raw = yf.download(t, start=START, end=END,
                           auto_adjust=True, progress=False)
        s = (raw['Close'] if isinstance(raw.columns, pd.MultiIndex)
             else raw.iloc[:,0]).squeeze()
        if s.notna().mean() > 0.3: dfs[t] = s
        print(f'  {t}: {s.notna().sum()} days')
    except Exception as e: print(f'  {t}: {e}')

prices  = pd.DataFrame(dfs).ffill(limit=3).replace(0,np.nan)
log_ret = np.log(prices/prices.shift(1)).clip(-0.20,0.20)
bm_ret  = log_ret[BENCHMARK].dropna()
strat   = [c for c in log_ret.columns
           if c != BENCHMARK and not c.startswith('^') and '=' not in c]
ew_ret  = log_ret[strat].mean(axis=1).reindex(bm_ret.index).fillna(0)
print(f'\nEW strategy: {len(strat)} assets | {len(ew_ret)} days')

## 3. Kill-Switch Features & Labels (Shrinking Horizon)

In [ ]:
REBAL_DAYS = 63
feat = pd.DataFrame(index=ew_ret.index)
feat['vol_10d']  = ew_ret.rolling(10).std() * np.sqrt(252)
feat['vol_21d']  = ew_ret.rolling(21).std() * np.sqrt(252)
feat['vol_63d']  = ew_ret.rolling(63).std() * np.sqrt(252)
feat['vol_ratio']= feat['vol_10d'] / feat['vol_63d'].replace(0,np.nan)
feat['mom_5d']   = ew_ret.rolling(5).sum()
feat['mom_21d']  = ew_ret.rolling(21).sum()
feat['bm_lag1']  = bm_ret.shift(1)
cum  = (1+ew_ret.fillna(0)).cumprod()
peak = cum.cummax()
feat['dd']       = (cum - peak) / peak

# Add cross-asset features
for col, r in [('usdinr',log_ret.get('USDINR=X')),
               ('spy',log_ret.get('SPY')),
               ('gold',log_ret.get('GC=F'))]:
    if r is not None:
        feat[f'{col}_5d'] = r.rolling(5).sum().reindex(feat.index)

# Shrinking-horizon label
dd_ser = feat['dd']
n = len(dd_ser)
labels = np.zeros(n, dtype=int)
for i in range(n):
    day_in_cycle = i % REBAL_DAYS
    days_left    = REBAL_DAYS - day_in_cycle
    fwd_end      = min(i+1+days_left, n)
    if fwd_end > i+1:
        if (dd_ser.iloc[i+1:fwd_end] < DD_THRESHOLD).any():
            labels[i] = 1

feat = feat.replace([np.inf,-np.inf],np.nan).fillna(0)
print(f'Features: {feat.shape[1]}  |  Label positive rate: {labels.mean():.3f}')

## 4. RF Kill-Switch (Walk-Forward)

In [ ]:
TRAIN_W, TEST_W = 756, 63
feat_arr = feat.values
n = len(feat_arr)
ks_proba = np.full(n, 0.0)

for start in range(TRAIN_W, n-TEST_W, TEST_W):
    Xtr = feat_arr[start-TRAIN_W:start]; ytr = labels[start-TRAIN_W:start]
    Xte = feat_arr[start:start+TEST_W]
    if ytr.sum() < 5: continue
    rf = RandomForestClassifier(n_estimators=200, max_depth=6,
                                 class_weight='balanced', random_state=42, n_jobs=-1)
    rf.fit(Xtr, ytr)
    ks_proba[start:start+TEST_W] = rf.predict_proba(Xte)[:,1]

ks_ser = pd.Series(ks_proba, index=feat.index, name='ks_prob')
print(f'Kill-switch signal: mean={ks_ser.mean():.4f}  max={ks_ser.max():.4f}')
print(f'Days above hard_stop ({HARD_STOP_P}): {(ks_ser >= HARD_STOP_P).sum()}')

## 5. Two-Zone DRL Position Sizing

In [ ]:
rf_daily = RF_ANNUAL / 252
cost     = COST_BPS / 10_000
weights, w_t = [], 1.0
cum_nav  = 1.0; peak_nav = 1.0

for i, (dt, ret) in enumerate(ew_ret.items()):
    cum_nav  = max(cum_nav * (1+ret), 1e-8)
    peak_nav = max(peak_nav, cum_nav)
    dd_t     = (cum_nav - peak_nav) / peak_nav
    ks_p     = float(ks_ser.get(dt, 0.0))

    if ks_p >= HARD_STOP_P:
        w_t = W_MIN
    elif ks_p < DEAD_ZONE:
        # Zone 1: ignore mild signals — stay invested
        grad = (ret - rf_daily) - DRL_LAMBDA*(1 if dd_t < DD_THRESHOLD else 0)
        w_t  = float(np.clip(w_t + 0.05*np.sign(grad), W_MIN, 1.0))
    else:
        # Zone 2: linear soft reduction
        zone_pos = (ks_p - DEAD_ZONE) / (HARD_STOP_P - DEAD_ZONE)
        target_w = 1.0 - zone_pos*(1.0 - W_MIN)
        grad = (ret - rf_daily) - DRL_LAMBDA*(1 if dd_t < DD_THRESHOLD else 0)
        w_t  = float(np.clip(w_t + 0.05*np.sign(grad), W_MIN, target_w))
    weights.append(w_t)

w_ser   = pd.Series(weights, index=ew_ret.index)
dw      = w_ser.diff().abs().fillna(0)
mgd_ret = ew_ret * w_ser - dw * cost
print(f'Mean weight: {w_ser.mean():.3f}  |  Days at floor: {(w_ser<=W_MIN+0.01).sum()}')

## 6. Performance Comparison & H0 Test

In [ ]:
def metrics(r, rf=RF_ANNUAL, name=''):
    r = r.dropna()
    ann = r.mean()*252; vol = r.std()*np.sqrt(252)
    sr  = (ann-rf)/vol if vol>0 else 0
    cum = (1+r).cumprod(); mdd = ((cum-cum.cummax())/cum.cummax()).min()
    cal = ann/abs(mdd) if mdd<0 else 0
    hit = (r>0).mean()
    print(f'{name:<20} ret={ann:.2%}  vol={vol:.2%}  SR={sr:.3f}  '
          f'MDD={mdd:.2%}  Calmar={cal:.3f}  Hit={hit:.2%}')
    return dict(ann_ret=ann,ann_vol=vol,sharpe=sr,mdd=mdd,calmar=cal)

print('Performance Comparison:')
m_bm  = metrics(bm_ret.reindex(mgd_ret.index), name='Benchmark (Nifty)')
m_ew  = metrics(ew_ret.reindex(mgd_ret.index), name='Unmanaged EW')
m_mgd = metrics(mgd_ret, name='Managed (KS+DRL)')

mdd_red = (abs(m_bm['mdd']) - abs(m_mgd['mdd'])) / abs(m_bm['mdd'])
n = min(len(mgd_ret),len(bm_ret.reindex(mgd_ret.index)))
a = mgd_ret.dropna().values; b = bm_ret.reindex(mgd_ret.index).dropna().values
n2 = min(len(a),len(b)); a,b = a[-n2:],b[-n2:]
lev_stat, lev_p = stats.levene(a, b)
t_stat, t_p     = stats.ttest_ind(a, b)

print(f'\nMDD reduction  : {mdd_red:.2%}  (threshold ≥15%: {"PASS ✓" if mdd_red>=0.15 else "FAIL"})')
print(f't-test         : t={t_stat:.4f}  p={t_p:.4f}  (note: inappropriate for scaling strategy)')
print(f'Levene (var)   : stat={lev_stat:.4f}  p={lev_p:.6f}  {"PASS ✓" if lev_p<0.05 else "fail"}')
print(f'Calmar ≥ 0.5   : {"PASS ✓" if m_mgd["calmar"]>=0.5 else "FAIL"} ({m_mgd["calmar"]:.4f})')
reject = (mdd_red>=0.15) and (t_p<0.05 or lev_p<0.05) and (m_mgd['calmar']>=0.5)
print(f'\nFINAL: {"REJECT H0 ✓" if reject else "FAIL TO REJECT H0"}')

## 7. Portfolio Comparison Chart

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8),
                                 gridspec_kw={'height_ratios':[3,1]})
bm_cum  = (1+bm_ret.reindex(mgd_ret.index).fillna(0)).cumprod()
mgd_cum = (1+mgd_ret.fillna(0)).cumprod()

ax1.plot(bm_cum.index, bm_cum.values, color='#607080', lw=1.2,
         label='Benchmark (Nifty 50)')
ax1.plot(mgd_cum.index, mgd_cum.values, color='#1D9E75', lw=1.8,
         label='Managed (Two-zone KS+DRL)')
ax1.set_yscale('log'); ax1.legend(fontsize=10)
ax1.set_title('RQ4 India — Managed vs Benchmark Portfolio', fontweight='bold', fontsize=12)
ax1.set_ylabel('Portfolio Value (log scale)')
ax1.grid(alpha=0.25)

ax2.plot(w_ser.index, w_ser.values, color='#1C3678', lw=0.8)
ax2.fill_between(w_ser.index, w_ser.values, W_MIN, alpha=0.25, color='#1C3678')
ax2.axhline(DEAD_ZONE, ls='--', lw=0.8, color='#EF9F27', label=f'Dead zone ({DEAD_ZONE})')
ax2.axhline(HARD_STOP_P, ls='--', lw=0.8, color='#E24B4A', label=f'Hard stop ({HARD_STOP_P})')
ax2.set_ylabel('Portfolio Weight'); ax2.legend(fontsize=8)
ax2.set_ylim(0,1.05); ax2.grid(alpha=0.25)

plt.tight_layout()
plt.savefig('../results/figures/06_rq4_managed_vs_bm.png', dpi=150)
plt.show(); print('RQ4 chart saved')